# Registro documental: ONNX → HEF para Jolkan-Baalam

Notebook exclusivamente documental. No ejecuta celdas de compilación ni modifica el ONNX original.

**Modelo:** `mau-cr/mayan_best_model`  
**Hardware objetivo:** Hailo-10H en Raspberry Pi 5  
**Entorno de conversión:** Google Colab, DFC 5.4.0, Python 3.10


## Cómo usar este registro

La notebook de trabajo es `compile_mayan_asr_hailo_organizada.ipynb`. Esta notebook explica qué se intentó, por qué se hizo, qué se validó y qué falta.

En cada sesión se añade una sección al final con fecha, entorno, candidato, resultado, errores y siguiente paso. Se separan siempre **Confirmado**, **Hipótesis** y **Pendiente**.


## Flujo completo

```text
ONNX + datos externos
  → transformación de operaciones incompatibles
  → equivalencia numérica
  → parseo DFC
  → bloques Transformer adicionales
  → calibración y cuantización
  → HEF
  → HailoRT en Raspberry Pi
```

Parseo, equivalencia y ejecución son validaciones distintas. Pasar una no demuestra las siguientes.


## Modelo y contrato experimental

- Entrada original: `[batch, sequence_length]`.
- Prueba experimental: `[1,64000]`, aproximadamente cuatro segundos a 16 kHz.
- Salida del ASR: `[batch, tiempo, 38]`.
- El encoder produce aproximadamente 199 posiciones.
- El candidato del Transformer 0 usa entrada y salida `[1,199,1280]`.
- CTC y KenLM permanecen fuera de Hailo en esta fase.


## Sesión 2026-09-16 — estado confirmado

### Confirmado

- El parser del grafo original falla en `/wav2vec2/feature_extractor/Unsqueeze`.
- Las siete convoluciones Conv1D del extractor fueron representadas como `Reshape → Conv2D → Reshape`.
- El parser Hailo acepta el extractor convertido.
- La equivalencia observada del extractor tiene error máximo aproximado `1.89e-4` y error medio `9.61e-6`.
- La convolución posicional fue adaptada a Conv2D con pesos `weight_norm` materializados como initializer estático.
- La equivalencia observada de los pesos posicionales tiene error máximo aproximado `4.70e-5` y error medio `2.11e-6`.
- El candidato integrado del Transformer 0 —atención, FFN/GELU, residuales y adapter YUA— pasa el parser DFC.

### Decisiones

- Mantener el ONNX original sin sobrescribirlo.
- Usar formas estáticas para la ventana experimental `[1,199,1280]`.
- Resolver el layout de la convolución posicional con `Reshape` y Conv2D.
- Mantener el decoder CTC/KenLM fuera de Hailo inicialmente.


## Por qué se transformó el grafo

Hailo no acepta automáticamente cualquier grafo publicado en ONNX o PyTorch. En este caso:

- Las Conv1D se convierten a Conv2D usando una dimensión espacial de tamaño uno.
- La convolución posicional tenía pesos dinámicos derivados de `weight_norm`; se materializan una vez para eliminar esa subred del candidato.
- El `Transpose` 3D de la convolución posicional se evita porque provocaba un fallo de memoria del parser; la atención conserva los `Transpose` aceptados en el candidato del Transformer 0.

Estas transformaciones deben acompañarse de una comparación numérica contra el modelo original.


## Estado hacia un HEF válido

### Pendiente inmediato

1. Comparar numéricamente el Transformer 0 completo contra el grafo original.
2. Encadenar más bloques Transformer y registrar el límite de memoria del DFC.
3. Integrar la salida CTC y fijar el reparto CPU/Hailo.
4. Preparar datos de calibración autorizados.
5. Ejecutar calibración, cuantización y compilación a HEF.
6. Cargar el HEF con HailoRT y probarlo en Raspberry Pi.
7. Medir WER, latencia, RTF y memoria.

### Criterio de validez

No se llamará válido a un HEF solo porque se genere sin error. Debe cargar con HailoRT, producir salidas comparables al modelo de referencia y ejecutarse en el Hailo-10H objetivo con métricas registradas.


## Plantilla para la siguiente sesión

### Sesión YYYY-MM-DD — título

**Entorno:**  
**Candidato:**  
**Objetivo:**

### Confirmado

- 

### Hipótesis

- 

### Resultado observado

- 

### Errores y limitaciones

- 

### Siguiente paso

- 


## Sesión 2026-09-16 — equivalencia numérica del Transformer 0

### Confirmado

La validación ejecutó el subgrafo Transformer 0 original y el candidato completo con la misma entrada determinista `[1,199,1280]`.

- Referencia: `(1, 199, 1280)`.
- Candidato: `(1, 199, 1280)`.
- Error absoluto máximo: `0.00000000e+00`.
- Error absoluto medio: `0.00000000e+00`.
- Error relativo máximo: `0.00000000e+00`.
- Resultado: `EQUIVALENCIA OK: Transformer 0`.

### Interpretación

El candidato extraído conserva exactamente la salida del subgrafo Transformer 0 de referencia para esta prueba. Esto valida la transformación estructural antes de intentar encadenar más bloques o cuantizar.

### Limitación

La prueba todavía es FP32 y se ejecutó con ONNX Runtime en el entorno de Colab. No demuestra aún equivalencia cuantizada, generación de HEF ni ejecución en HailoRT.

### Siguiente paso

Intentar encadenar el Transformer 1, repetir la validación numérica y registrar el consumo de memoria y el resultado del parser DFC.


## Sesión 2026-09-16 — equivalencia numérica de Transformers 0 y 1

### Confirmado

La validación ejecutó los dos bloques Transformer encadenados y el subgrafo equivalente del ONNX original con la misma entrada determinista `[1,199,1280]`.

- Referencia: `(1, 199, 1280)`.
- Candidato: `(1, 199, 1280)`.
- Error absoluto máximo: `0.00000000e+00`.
- Error absoluto medio: `0.00000000e+00`.
- Error relativo máximo: `0.00000000e+00`.
- Resultado: `EQUIVALENCIA OK: Transformers 0 y 1`.

### Interpretación

La extensión del candidato a dos bloques conserva exactamente la salida del ONNX de referencia para esta prueba FP32. El parser DFC también aceptó el candidato de ambos bloques.

### Limitación

Todavía no se ha probado cuantización, generación de HEF, HailoRT ni ejecución en Raspberry Pi. La entrada continúa siendo una ventana estática `[1,199,1280]`.

### Siguiente paso

Añadir el Transformer 2, repetir parseo y equivalencia numérica, y registrar el consumo de memoria del DFC.


## Sesión 2026-09-17 — encoder completo hasta logits CTC

### Confirmado

El candidato `mayan_encoder48_logits` integra los 48 bloques Transformer, la normalización final del encoder y la cabeza CTC (`lm_head`) hasta `logits`.

- Entrada: activaciones `[1,199,1280]`.
- Salida: logits `[1,199,38]`.
- El parser de Hailo-10H aceptó previamente los 48 Transformers.
- La validación FP32 comparó referencia y candidato en cuatro puntos: salida Transformer, normalización final, salida de `lm_head/MatMul` y logits.
- Error absoluto máximo, medio y relativo máximo: `0.0` en los cuatro puntos.
- Resultado: `EQUIVALENCIA OK: encoder de 48 Transformers hasta logits`.

### Decisión de implementación

Los pesos externos de los 48 Transformers se conservan en su archivo `model.onnx_data` mediante un hardlink local en el directorio del candidato. Los pesos finales se guardan como archivos externos locales. Esto evita reserializar los pesos del Transformer, que había introducido una divergencia FP32.

### Limitación

El candidato aún recibe activaciones del encoder; no recibe audio. Falta integrar el extractor Conv2D, la proyección de características, la convolución posicional estática y la suma que alimenta al Transformer 0. CTC/KenLM permanece fuera de Hailo.

### Siguiente paso

Construir y validar el candidato integrado desde audio `[1,64000]` hasta logits `[1,199,38]`, usando los componentes ya validados y preservando las rutas de pesos externos.


## Cierre de sesión 2026-09-17 — qué está completo y qué falta

### Confirmado

```text
activaciones [1,199,1280]
  → 48 Transformers con adapters YUA
  → LayerNormalization final
  → lm_head
  → logits [1,199,38]
```

- El parser DFC acepta el candidato de los 48 Transformers.
- El candidato completo del encoder hasta logits tiene equivalencia FP32 exacta frente al subgrafo del ONNX original: error absoluto máximo, medio y relativo máximo `0.0`.
- El extractor Conv1D→Conv2D y la convolución posicional estática ya cuentan con parseo y validaciones parciales documentadas.

### Aún no completo

```text
audio [1,64000]
  → extractor Conv2D
  → proyección de características
  → convolución posicional estática
  → encoder48→logits
```

La integración anterior aún no existe como un candidato único. Por eso no hay HEF, inferencia HailoRT, WER, latencia, RTF ni prueba en Raspberry Pi.

### Punto de inicio para la próxima sesión

1. Inspeccionar y reutilizar los candidatos ya validados del extractor y de la convolución posicional.
2. Construir un candidato nuevo desde audio `[1,64000]` hasta logits `[1,199,38]`.
3. Validarlo numéricamente contra el ONNX original antes de iniciar calibración.

### Nota sobre pesos externos

El candidato `encoder48→logits` debe conservar los pesos de los Transformers mediante el archivo externo existente dentro de su directorio de candidato. Cargar y reserializar todos esos pesos cambió la salida FP32; la implementación correcta usa un hardlink local y pesos externos locales para la cabeza CTC.

